# 🚀 Фабрика нейросетей & Оптимизатор ансамбля PRL/USDT (Google Colab / GPU T4)

Данный ноутбук реализует исследовательский конвейер глубокого обучения на базе архитектур из `_tools/ltsm_core/model_builder.py` и оптимизатора ансамблей `_tools/ensemble_predictor.py`:

### 🔬 4 специализированные архитектуры + LightGBM:
1. **`conv1d+gru` (Temporal Anchor):** GaussianNoise + LayerNorm + Conv1D Bottleneck + Causal Conv1D + GRU(64) + Dense(64, GELU). Отслеживает динамику OFI, скрытое накопление и айсберг-заявки.
2. **`causal_cnn` (Causal ResNet):** Глубокая остаточная сверточная сеть со строгим `causal`-паддингом и skip-connections. Извлекает геометрию стакана без заглядывания в будущее.
3. **`attention` (Transformer Multi-Head Attention):** 4 головы самовнимания (`key_dim=16`) + Residual Connection + GlobalAveragePooling1D. Улавливает нелокальные временные связи и опережающие импульсы BigONE за 5–15 секунд.
4. **`mlp_bottleneck` (Spatial Dropout Denoiser):** Выбивает `SpatialDropout1D(0.2)` — случайные 20% каналов признаков целиком во времени, защищая от переобучения на доминирующем индикаторе стакана + Residual Dense блоки.

### 🛡️ Защита от переобучения и калибровка вероятностей:
- **Label Smoothing (0.06):** устраняет излишнюю самоуверенность Softmax и калибрует вероятности нейросетей для слияния с резкими порогами деревьев LightGBM.
- **AdamW + Weight Decay (1e-4) + Gradient Clipping (`clipnorm=1.0`):** защита от взрыва градиентов на проливах.
- **Cosine Decay / ReduceLROnPlateau + EarlyStopping:** плавная посадка шага обучения в глобальный минимум.
- **Математический подбор весов (SLSQP + комбинаторный поиск):** находит точные веса альянса $W^* = [w_{\text{lgb}}, w_{\text{gru}}, w_{\text{cnn}}, w_{\text{attn}}, w_{\text{mlp}}]$, максимизирующие OOS IC и минимизирующие Log-Loss.

In [1]:
# 1. Проверка GPU NVIDIA T4 и включение аппаратного ускорения Mixed Precision (FP16)
!nvidia-smi

import os
import gc
import time
import json
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, Conv1D, GaussianNoise,
    GRU, Add, Activation, Flatten, GlobalAveragePooling1D,
    MultiHeadAttention, SpatialDropout1D
)
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.utils.class_weight import compute_class_weight

tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("✅ Версия TensorFlow:", tf.__version__)
print("✅ Доступные GPU:", tf.config.list_physical_devices("GPU"))
print("✅ Политика Mixed Precision:", tf.keras.mixed_precision.global_policy().name)

Mon Oct 19 09:18:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os
import gc
import time
import json
import itertools
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, Conv1D, GaussianNoise,
    GRU, Add, Activation, Flatten, GlobalAveragePooling1D,
    MultiHeadAttention, SpatialDropout1D
)
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, log_loss
from sklearn.utils.class_weight import compute_class_weight
tf.keras.mixed_precision.set_global_policy("mixed_float16")

# 2. Загрузка подготовленного датасета prl_neural_colab_data.npz (27 МБ, H=5, 12, 25)
FILE_ID = "169R817rVSz_TENn_QW3nue1-K1tLinjC"
DATA_FILE = "prl_neural_colab_data.npz"

for candidate in [
    "data/prl_neural_colab_data.npz",
    "prl_neural_colab_data.npz",
    "/content/prl_neural_colab_data.npz",
    "/content/drive/MyDrive/prl_neural_colab_data.npz",
    "/content/drive/MyDrive/crypto_bot/prl_neural_colab_data.npz"
]:
    if os.path.exists(candidate):
        DATA_FILE = candidate
        break

need_download = not os.path.exists(DATA_FILE)
if not need_download:
    try:
        _chk = np.load(DATA_FILE, allow_pickle=True)
        if "y_train_h25" not in _chk.files:
            print("🔄 Обнаружена старая версия датасета без H=25с, скачиваем обновленную...")
            need_download = True
        del _chk
    except Exception:
        need_download = True

if need_download:
    print("☁️ Автоматическое скачивание prl_neural_colab_data.npz с Google Drive...")
    import gdown
    DATA_FILE = "prl_neural_colab_data.npz"
    gdown.download(f"https://drive.google.com/uc?id={FILE_ID}", DATA_FILE, quiet=False)

raw = np.load(DATA_FILE, allow_pickle=True)
X_tr = raw["X_train"]
X_te = raw["X_test"]

# Строго по закону проекта L = H * K (из train_rolling_swarm.py):
# H=25с -> K=3.0 (Lookback L=75с) | H=12с -> K=3.5 (Lookback L=42с) | H=5с -> K=3.0 (Lookback L=15с)
HORIZON = "5"
if HORIZON == "25":
    K_RATIO = 3.0
    SEQ_LEN = int(round(25 * K_RATIO))  # 75 секунд!
    y_tr = raw["y_train_h25"]
    y_te = raw["y_test_h25"]
    dt_te = raw["dt_test_h25"]
elif HORIZON == "12":
    K_RATIO = 3.5
    SEQ_LEN = int(round(12 * K_RATIO))  # 42 секунды!
    y_tr = raw["y_train_h12"]
    y_te = raw["y_test_h12"]
    dt_te = raw["dt_test_h12"]
else:
    K_RATIO = 3.0
    SEQ_LEN = int(round(5 * K_RATIO))   # 15 секунд!
    y_tr = raw["y_train_h5"]
    y_te = raw["y_test_h5"]
    dt_te = raw["dt_test_h5"]

ts_tr = raw["ts_train"]
ts_te = raw["ts_test"]
feature_names = raw["features"]

print(f"✅ Train: {X_tr.shape} ({ts_tr[0]} ➔ {ts_tr[-1]})")
print(f"✅ OOS Test: {X_te.shape} ({ts_te[0]} ➔ {ts_te[-1]})")
print(f"✅ Признаков: {len(feature_names)} | Горизонт H={HORIZON}с | K={K_RATIO} | Lookback L={SEQ_LEN}с")

☁️ Автоматическое скачивание prl_neural_colab_data.npz с Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=169R817rVSz_TENn_QW3nue1-K1tLinjC
From (redirected): https://drive.google.com/uc?id=169R817rVSz_TENn_QW3nue1-K1tLinjC&confirm=t&uuid=d222695d-6cb5-4a8b-b531-90481d0c093b
To: /content/prl_neural_colab_data.npz
100%|██████████| 28.4M/28.4M [00:00<00:00, 66.9MB/s]


✅ Train: (210000, 299) (2026-10-15 23:01:18+03:00 ➔ 2026-10-18 09:21:17+03:00)
✅ OOS Test: (20778, 299) (2026-10-18 09:27:30+03:00 ➔ 2026-10-18 15:13:47+03:00)
✅ Признаков: 299 | Горизонт H=5с | K=3.0 | Lookback L=15с


In [3]:
# 3. GPU-резидентные 3D-генераторы (без упора в слабый CPU Colab) + Мониторинг видеоядра T4
import subprocess

BATCH_SIZE = 2048
N_CLASSES = 7

# Нормализация строго по Train
mean = np.nanmean(X_tr, axis=0)
std = np.nanstd(X_tr, axis=0) + 1e-6

X_tr_norm = np.clip((X_tr - mean) / std, -5.0, 5.0).astype(np.float16)
X_te_norm = np.clip((X_te - mean) / std, -5.0, 5.0).astype(np.float16)

y_tr_oh = tf.keras.utils.to_categorical(y_tr, num_classes=N_CLASSES).astype(np.float32)
y_te_oh = tf.keras.utils.to_categorical(y_te, num_classes=N_CLASSES).astype(np.float32)

base_weights = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr[SEQ_LEN - 1:])
custom_weights = dict(enumerate(base_weights))
custom_weights[0] *= 1.35
custom_weights[1] *= 1.25
custom_weights[2] *= 1.10

# Размещаем плоские матрицы (125 МБ) прямо в видеопамяти GPU T4!
with tf.device('/GPU:0'):
    X_tr_gpu = tf.constant(X_tr_norm, dtype=tf.float16)
    y_tr_gpu = tf.constant(y_tr_oh[SEQ_LEN - 1:], dtype=tf.float32)
    X_te_gpu = tf.constant(X_te_norm, dtype=tf.float16)
    y_te_gpu = tf.constant(y_te_oh[SEQ_LEN - 1:], dtype=tf.float32)
    offsets_gpu = tf.range(SEQ_LEN, dtype=tf.int32)[tf.newaxis, :]

def make_gpu_sequence_dataset(X_gpu, y_gpu, seq_len=75, batch_size=2048, shuffle=True):
    n_win = int(X_gpu.shape[0]) - seq_len + 1
    ds = tf.data.Dataset.from_tensor_slices(tf.range(n_win, dtype=tf.int32))
    if shuffle:
        ds = ds.shuffle(buffer_size=n_win, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(
        lambda b_idx: (tf.gather(X_gpu, b_idx[:, tf.newaxis] + offsets_gpu), tf.gather(y_gpu, b_idx)),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_gpu_sequence_dataset(X_tr_gpu, y_tr_gpu, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, shuffle=True)
val_ds = make_gpu_sequence_dataset(X_te_gpu, y_te_gpu, seq_len=SEQ_LEN, batch_size=BATCH_SIZE, shuffle=False)

y_true_oos = y_te[SEQ_LEN - 1:]
y_true_oos_oh = y_te_oh[SEQ_LEN - 1:]
dt_true_oos = dt_te[SEQ_LEN - 1:]

# Колбэк мониторинга загрузки видеоядра GPU для VS Code
class GPUMonitorCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        try:
            out = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=utilization.gpu,utilization.memory,memory.used,power.draw,temperature.gpu",
                 "--format=csv,noheader,nounits"], text=True
            ).strip().split(", ")
            print(f"   ⚡ [GPU T4] Ядро: {out[0]}% | Шина VRAM: {out[1]}% | Память: {out[2]} MiB | Мощность: {out[3]}W | Темп: {out[4]}°C")
        except Exception:
            pass

print(f"✅ GPU-генераторы готовы в VRAM! OOS проверочных секунд: {len(y_true_oos):,}")

✅ GPU-генераторы готовы в VRAM! OOS проверочных секунд: 20,764


In [4]:
# 4. Фабрика 4 нейросетевых архитектур (из _tools/ltsm_core/model_builder.py)
def build_conv1d_gru(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Conv1D(64, 3, padding='causal', activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.1)(x)
    x = GRU(64, return_sequences=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="conv1d_gru")

def _res_block(x, filters, kernel_size, dilation, l2_reg):
    sc = x
    r = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation, kernel_regularizer=regularizers.l2(l2_reg))(x)
    r = LayerNormalization()(r)
    r = Activation('gelu')(r)
    r = Dropout(0.1)(r)
    r = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation, kernel_regularizer=regularizers.l2(l2_reg))(r)
    r = LayerNormalization()(r)
    if int(sc.shape[-1]) != filters:
        sc = Conv1D(filters, 1, padding='valid', kernel_regularizer=regularizers.l2(l2_reg))(sc)
    return Activation('gelu')(Add()([sc, r]))

def build_causal_cnn(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = _res_block(x, 32, 3, 1, l2_reg)
    x = _res_block(x, 64, 3, 2, l2_reg)
    x = _res_block(x, 64, 3, 4, l2_reg)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="causal_cnn")

def build_attention(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    attn = MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.2)(x, x)
    x = LayerNormalization()(Add()([x, attn]))
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="attention")

def build_mlp_bottleneck(seq_len, n_features, n_classes=7, l2_reg=1e-4):
    inp = Input(shape=(seq_len, n_features), name="input_layer")
    x = GaussianNoise(0.01)(inp)
    x = LayerNormalization()(x)
    x = SpatialDropout1D(0.20)(x)
    x = Conv1D(32, 1, activation='gelu', kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Flatten()(x)
    x = Dense(128, kernel_regularizer=regularizers.l2(l2_reg * 3))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.20)(x)
    sc = x
    x = Dense(128, kernel_regularizer=regularizers.l2(l2_reg * 3))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.20)(x)
    x = Add()([sc, x])
    x = Dense(64, kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = LayerNormalization()(x)
    x = Activation('gelu')(x)
    x = Dropout(0.15)(x)
    out = Dense(n_classes, activation='softmax', dtype='float32')(x)
    return Model(inp, out, name="mlp_bottleneck")

MODEL_FACTORY = {
    "conv1d+gru": build_conv1d_gru,
    "causal_cnn": build_causal_cnn,
    "attention": build_attention,
    "mlp_bottleneck": build_mlp_bottleneck
}
print("✅ Фабрика из 4 архитектур инициализирована:", list(MODEL_FACTORY.keys()))

✅ Фабрика из 4 архитектур инициализирована: ['conv1d+gru', 'causal_cnn', 'attention', 'mlp_bottleneck']


In [5]:
# 5. Метрики Out-of-Sample и обучение базового эксперта LightGBM (со взвешенными состояниями)
import time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

if "X_tr" not in globals() or "y_true_oos" not in globals():
    raise RuntimeError("❌ Сначала запустите ячейки 1, 2, 3 и 4 сверху вниз (или нажмите Ctrl+F9 / Выполнить все)!")

CLASSES_VEC = np.array([-3, -2, -1, 0, 1, 2, 3], dtype=np.float32)

def evaluate_oos_probs(probs, y_true=y_true_oos, y_oh=y_true_oos_oh, dt_ticks=dt_true_oos):
    p_clip = np.clip(probs, 1e-7, 1.0 - 1e-7)
    p_clip = p_clip / np.sum(p_clip, axis=1, keepdims=True)
    ez = np.dot(p_clip, CLASSES_VEC)
    ic = float(np.corrcoef(ez, dt_ticks)[0, 1]) if np.std(ez) > 1e-6 else 0.0
    try:
        auc = float(roc_auc_score(y_true, p_clip, multi_class="ovr"))
    except Exception:
        auc = 0.5
    ll = float(-np.mean(np.sum(y_oh * np.log(p_clip), axis=1)))
    top2_th = np.quantile(ez, 0.98)
    top2_mask = ez >= top2_th
    top2_pnl = float(np.mean(dt_ticks[top2_mask])) if np.sum(top2_mask) > 0 else 0.0
    return {"ic": ic, "auc": auc, "log_loss": ll, "top2_pnl": top2_pnl}

print("🌲 Обучение референсного эксперта LightGBM (со взвешенными состояниями custom_weights)...")
t0 = time.time()
sample_w_tr = np.array([custom_weights[int(c)] for c in y_tr], dtype=np.float32)
dtrain = lgb.Dataset(X_tr, label=y_tr, weight=sample_w_tr)
lgb_params = {
    "objective": "multiclass", "num_class": 7, "learning_rate": 0.08,
    "num_leaves": 31, "max_depth": 6, "min_child_samples": 50,
    "subsample": 0.8, "subsample_freq": 1, "colsample_bytree": 0.8,
    "n_jobs": 4, "verbose": -1, "seed": 42
}
lgb_model = lgb.train(lgb_params, dtrain, num_boost_round=70)
probs_lgb = lgb_model.predict(X_te[SEQ_LEN - 1:])
del dtrain
gc.collect()

oos_predictions = {"lightgbm": probs_lgb}
leaderboard = []
m_lgb = evaluate_oos_probs(probs_lgb)
leaderboard.append({"model": "lightgbm", "params": 0, "time_s": round(time.time() - t0, 1), **m_lgb})
print(f"✅ LightGBM готов ({leaderboard[0]['time_s']}с) | IC: {m_lgb['ic']:+.4f} | AUC: {m_lgb['auc']:.4f} | Loss: {m_lgb['log_loss']:.4f} | Top-2% PnL: {m_lgb['top2_pnl']:+.2f} тиков")

🌲 Обучение референсного эксперта LightGBM (со взвешенными состояниями custom_weights)...
✅ LightGBM готов (87.8с) | IC: +0.1602 | AUC: 0.8026 | Loss: 1.1502 | Top-2% PnL: +0.05 тиков


In [6]:
# 6. Многократное обучение (10 ранов) с унаследованными механизмами:
#    SmartBacktrackCallback (откат весов + сброс инерции AdamW) +
#    ElasticPatienceProfiler (эластичное терпение + Z-Score гильотина, сжатая в 5 раз: 10 эпох вместо 50, срез на 5-й эпохе)
import sqlite3
import shutil

ACTIVE_ARCHS = ["conv1d+gru", "causal_cnn", "attention", "mlp_bottleneck"]
RUNS_PER_ARCH = 10       # Ограничение: 10 ранов на каждую архитектуру
MAX_EPOCHS = 10          # В 5 раз меньше базового лимита (50 / 5 = 10 эпох)
HARD_PRUNE_EPOCH = 5     # В 5 раз меньше базовой отсечки гильотины (25 / 5 = 5 эпоха)
TOP_K_PER_ARCH = 3       # Сохраняем Топ-3 элитных чекпоинта каждой архитектуры для супер-ансамбля
L2_REG = 1e-4

os.makedirs("trained_models", exist_ok=True)


class SmartBacktrackCallback(tf.keras.callbacks.Callback):
    """Откат к лучшим весам + хирургический сброс моментов AdamW при плато val_loss (из ltsm_core/callbacks.py)."""
    def __init__(self, best_weights_path, monitor_loss="val_loss", factor=0.5, patience=2, min_lr=1e-5, max_rollbacks=3):
        super().__init__()
        self.best_weights_path = str(best_weights_path)
        self.monitor_loss = monitor_loss
        self.factor = factor
        self.patience = patience
        self.min_lr = min_lr
        self.max_rollbacks = max_rollbacks
        self.wait = 0
        self.rollback_count = 0
        self.best_loss = np.inf

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get(self.monitor_loss) if logs else None
        if current_loss is None:
            return
        if current_loss < self.best_loss - 1e-4:
            self.best_loss = current_loss
            self.wait = 0
            self.rollback_count = 0
        else:
            self.wait += 1
            if self.wait >= self.patience:
                self.rollback_count += 1
                if self.rollback_count >= self.max_rollbacks:
                    print(f"   🛑 [Backtrack] Лимит откатов LR ({self.max_rollbacks}) исчерпан.")
                    self.model.stop_training = True
                    return
                old_lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
                new_lr = max(old_lr * self.factor, self.min_lr)
                if os.path.exists(self.best_weights_path):
                    self.model.load_weights(self.best_weights_path)
                if hasattr(self.model.optimizer, "variables"):
                    opt_vars = self.model.optimizer.variables() if callable(self.model.optimizer.variables) else self.model.optimizer.variables
                    for var in opt_vars:
                        vname = var.name.lower()
                        if len(var.shape) >= 1 and any(k in vname for k in ("momentum", "velocity", "exponential_moving")):
                            var.assign(tf.zeros_like(var))
                self.model.optimizer.learning_rate.assign(new_lr)
                self.wait = 0
                print(f"   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: {new_lr:.6f}")


class ElasticPatienceProfiler(tf.keras.callbacks.Callback):
    """Эластичный бюджет эпох + Z-Score отбраковка слабых инициализаций (масштабировано в 5 раз)."""
    def __init__(self, history_losses, max_epochs=10, bonus_ratio=0.25, min_delta=5e-4,
                 hard_prune_epoch=5, hard_prune_z=4.0, max_z_score=2.0):
        super().__init__()
        self.max_epochs = max_epochs
        self.pruned = False
        self.micro_patience = max(1, int(round(0.20 * max_epochs)))   # 2 эпохи
        self.macro_patience = max(4.0, float(0.45 * max_epochs))     # старт с 4.5 эпох, растет при успехах до 10
        self.macro_bonus = max(0.75, bonus_ratio * self.micro_patience)
        self.min_delta = min_delta
        self.micro_wait = 0
        self.local_best_loss = np.inf
        self.hard_prune_epoch = hard_prune_epoch
        self.hard_prune_z = hard_prune_z
        self.max_z_score = max_z_score

        if len(history_losses) >= 3:
            k = max(2, int(np.ceil(len(history_losses) * 0.35)))
            elite = np.sort(history_losses)[:k]
            self.mu = float(np.mean(elite))
            self.sigma = float(max(np.std(elite), 0.015))
        else:
            self.mu = 1.65
            self.sigma = 0.15

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get("val_loss") if logs else None
        if current_loss is None:
            return
        if current_loss < self.local_best_loss - self.min_delta:
            self.local_best_loss = current_loss
            self.micro_wait = 0
            self.macro_patience = min(float(self.max_epochs), self.macro_patience + self.macro_bonus)
        else:
            self.micro_wait += 1

        guillotine_limit = self.mu + self.hard_prune_z * self.sigma
        if (epoch + 1) >= self.hard_prune_epoch and self.local_best_loss > guillotine_limit:
            print(f"   🔪 [Hard Pruning] Эпоха {epoch+1}: лучший Loss {self.local_best_loss:.4f} > гильотины {guillotine_limit:.4f}. Итерация сброшена.")
            self.model.stop_training = True
            self.pruned = True
            return

        if self.micro_wait >= self.micro_patience and len( 
            [1] if self.mu < 1.60 else []
        ) > 0:
            z_score = (self.local_best_loss - self.mu) / self.sigma
            if z_score > self.max_z_score:
                print(f"   🔪 [Elite Z-Score] Нет прогресса {self.micro_patience} эп. Loss {self.local_best_loss:.4f} вне элиты (Z={z_score:.2f}). Итерация сброшена.")
                self.model.stop_training = True
                self.pruned = True
                return
            self.micro_wait = 0

        if (epoch + 1) >= int(np.ceil(self.macro_patience)):
            print(f"   ⏳ [Elastic EarlyStop] Достигнут эластичный потолок ({int(np.ceil(self.macro_patience))}/{self.max_epochs} эп.).")
            self.model.stop_training = True

# Инициализируем локальную SQLite БД оркестратора (как в SmartOrchestrator)
db_conn = sqlite3.connect("trained_models/trading_factory_colab.db")
db_conn.execute("""
    CREATE TABLE IF NOT EXISTS runs (
        run_id TEXT PRIMARY KEY, arch TEXT, run_idx INT,
        val_loss REAL, ic REAL, auc REAL, top2_pnl REAL, status TEXT, time_s REAL
    )
""")
db_conn.commit()

trained_keras_models = {}
elite_pool_predictions = {"lightgbm": probs_lgb}
leaderboard = [item for item in leaderboard if item["model"] == "lightgbm"]

for arch in ACTIVE_ARCHS:
    print("\n" + "=" * 78)
    print(f"🚀 СЕРИЯ ИЗ {RUNS_PER_ARCH} РАНОВ АРХИТЕКТУРЫ: {arch.upper()} (H={HORIZON}с, L={SEQ_LEN}с, MAX_EPOCHS={MAX_EPOCHS})")
    print("=" * 78)

    arch_history_losses = []
    arch_candidates = []  # список (val_loss, -ic, run_idx, keras_path, probs, metrics, elapsed, n_params)

    for run_idx in range(1, RUNS_PER_ARCH + 1):
        gc.collect()
        t_run = time.time()
        tf.keras.utils.set_random_seed(1000 * (ACTIVE_ARCHS.index(arch) + 1) + run_idx * 37)

        model = MODEL_FACTORY[arch](seq_len=SEQ_LEN, n_features=X_tr.shape[1], n_classes=N_CLASSES, l2_reg=L2_REG)
        n_params = model.count_params()
        optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-2, weight_decay=L2_REG, clipnorm=1.0)
        model.compile(
            optimizer=optimizer,
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.06),
            metrics=["accuracy"]
        )

        temp_w = f"trained_models/tmp_{arch.replace('+', '_')}_r{run_idx}.weights.h5"
        profiler = ElasticPatienceProfiler(
            arch_history_losses, max_epochs=MAX_EPOCHS,
            hard_prune_epoch=HARD_PRUNE_EPOCH, hard_prune_z=4.0, max_z_score=2.0
        )
        cbs = [
            tf.keras.callbacks.ModelCheckpoint(
                filepath=temp_w, save_weights_only=True, monitor="val_loss", mode="min", save_best_only=True, verbose=0
            ),
            SmartBacktrackCallback(best_weights_path=temp_w, monitor_loss="val_loss", factor=0.5, patience=2, min_lr=1e-5, max_rollbacks=3),
            tf.keras.callbacks.TerminateOnNaN(),
            profiler
        ]
        if run_idx in (1, RUNS_PER_ARCH):
            cbs.append(GPUMonitorCallback())

        model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCHS, class_weight=custom_weights, callbacks=cbs, verbose=0)
        elapsed = round(time.time() - t_run, 1)

        if os.path.exists(temp_w):
            model.load_weights(temp_w)
            os.remove(temp_w)

        probs = model.predict(val_ds, verbose=0)
        metrics = evaluate_oos_probs(probs)
        val_loss = metrics["log_loss"]
        status = "PRUNED" if profiler.pruned else "COMPLETED"
        arch_history_losses.append(val_loss)

        run_id = f"{arch}_h{HORIZON}_r{run_idx:02d}"
        db_conn.execute(
            "INSERT OR REPLACE INTO runs VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
            (run_id, arch, run_idx, val_loss, metrics["ic"], metrics["auc"], metrics["top2_pnl"], status, elapsed)
        )
        db_conn.commit()

        cand_path = f"trained_models/cand_{arch.replace('+', '_')}_r{run_idx:02d}.keras"
        model.save(cand_path)
        arch_candidates.append((val_loss, -metrics["ic"], run_idx, cand_path, probs, metrics, elapsed, n_params))

        best_so_far = min(c[0] for c in arch_candidates)
        mark = "🏆 РЕКОРД" if val_loss <= best_so_far else ("🔪 PRUNED" if profiler.pruned else "✅ OK")
        print(
            f"  [{arch.upper()} | Ран {run_idx:02d}/{RUNS_PER_ARCH}] {mark} ({elapsed}с) -> "
            f"OOS Loss: {val_loss:.4f} | IC: {metrics['ic']:+.4f} | AUC: {metrics['auc']:.4f} | Top-2% PnL: {metrics['top2_pnl']:+.2f} тик."
        )
        del model

    # Сортируем 10 ранов по лучшей комбинации (минимальный Log-Loss и максимальный IC)
    arch_candidates.sort(key=lambda x: (round(x[0], 3), x[1]))
    best_cand = arch_candidates[0]
    _, _, best_r_idx, best_c_path, best_probs, best_metrics, best_elapsed, best_params = best_cand

    # Сохраняем лучшую модель архитектуры как основной чекпоинт + Топ-K в пул супер-ансамбля
    save_path = f"trained_models/model_{arch.replace('+', '_')}_h{HORIZON}s.keras"
    shutil.copy2(best_c_path, save_path)
    trained_keras_models[arch] = save_path
    oos_predictions[arch] = best_probs

    # Усредняем Топ-3 лучших рана внутри самой архитектуры (внутриархитектурный рой!)
    top_k_cands = arch_candidates[:TOP_K_PER_ARCH]
    swarm_probs = np.mean([c[4] for c in top_k_cands], axis=0)
    swarm_metrics = evaluate_oos_probs(swarm_probs)
    oos_predictions[f"{arch}_top{TOP_K_PER_ARCH}"] = swarm_probs

    for rank_i, c_item in enumerate(top_k_cands, 1):
        elite_pool_predictions[f"{arch}#r{c_item[2]}"] = c_item[4]

    # Удаляем временные файлы не-топовых ранов
    for c_item in arch_candidates[TOP_K_PER_ARCH:]:
        if os.path.exists(c_item[3]):
            os.remove(c_item[3])

    total_arch_time = round(sum(c[6] for c in arch_candidates), 1)
    leaderboard.append({"model": f"{arch} (best r{best_r_idx})", "params": best_params, "time_s": total_arch_time, **best_metrics})
    leaderboard.append({"model": f"{arch} (swarm top-{TOP_K_PER_ARCH})", "params": best_params * TOP_K_PER_ARCH, "time_s": total_arch_time, **swarm_metrics})

    print("-" * 78)
    print(f"🌟 ИТОГ [{arch.upper()}]: Лучший ран #{best_r_idx} -> IC: {best_metrics['ic']:+.4f}, Loss: {best_metrics['log_loss']:.4f}")
    print(f"🐝 РОЙ ТОП-{TOP_K_PER_ARCH} [{arch.upper()}]:        -> IC: {swarm_metrics['ic']:+.4f}, Loss: {swarm_metrics['log_loss']:.4f}, Top-2% PnL: {swarm_metrics['top2_pnl']:+.2f} тик.")

print("\n🏆 ИТОГОВЫЙ ЛИДЕРБОРД МОДЕЛЕЙ И ВНУТРИАРХИТЕКТУРНЫХ РОЕВ (Out-of-Sample):")
df_lb = pd.DataFrame(leaderboard).sort_values(by="ic", ascending=False)
print(df_lb.to_string(index=False))



🚀 СЕРИЯ ИЗ 10 РАНОВ АРХИТЕКТУРЫ: CONV1D+GRU (H=5с, L=15с, MAX_EPOCHS=10)
   ⚡ [GPU T4] Ядро: 37% | Шина VRAM: 25% | Память: 647 MiB | Мощность: 40.00W | Темп: 64°C
   ⚡ [GPU T4] Ядро: 4% | Шина VRAM: 3% | Память: 647 MiB | Мощность: 40.37W | Темп: 66°C
   ⚡ [GPU T4] Ядро: 43% | Шина VRAM: 28% | Память: 647 MiB | Мощность: 45.47W | Темп: 67°C
   ⚡ [GPU T4] Ядро: 44% | Шина VRAM: 29% | Память: 647 MiB | Мощность: 51.12W | Темп: 68°C
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.005000
   ⚡ [GPU T4] Ядро: 26% | Шина VRAM: 18% | Память: 647 MiB | Мощность: 41.45W | Темп: 69°C
Batch 1: Invalid loss, terminating training
   ⚡ [GPU T4] Ядро: 15% | Шина VRAM: 10% | Память: 647 MiB | Мощность: 41.65W | Темп: 69°C
  [CONV1D+GRU | Ран 01/10] 🏆 РЕКОРД (32.7с) -> OOS Loss: 1.9852 | IC: +0.1349 | AUC: 0.7611 | Top-2% PnL: +0.01 тик.
   📉 [Backtrack] Откат к лучшим весам + сброс инерции AdamW -> новый LR: 0.005000
Batch 1: Invalid loss, terminating training
  [CONV1D+GRU

In [7]:
# 7. Математический оптимизатор ансамбля (Комбинаторика + SLSQP оптимизация весов)
model_names = list(oos_predictions.keys())
prob_list = [oos_predictions[m] for m in model_names]
M = len(model_names)

print(f"🤝 1. Комбинаторный поиск лучших связок из {M} моделей (по методу ensemble_predictor.py)...")
combo_results = []
for k in range(1, M + 1):
    for indices in itertools.combinations(range(M), k):
        blend = np.mean([prob_list[i] for i in indices], axis=0)
        res = evaluate_oos_probs(blend)
        combo_name = " + ".join([model_names[i] for i in indices])
        combo_results.append({"alliance": combo_name, "k": k, **res})

df_combos = pd.DataFrame(combo_results).sort_values(by=["ic", "auc"], ascending=False)
print("\nТОП-10 РАВНОВЕСНЫХ СВЯЗОК (по OOS IC):")
print(df_combos.head(10).to_string(index=False))

print("\n🧮 2. Непрерывная выпуклая оптимизация весов (Scipy SLSQP: минимизация Log-Loss + максимизация IC)...")
def ensemble_objective(weights):
    w = np.array(weights, dtype=np.float64)
    w = w / np.sum(w)
    blend = sum(w[i] * prob_list[i] for i in range(M))
    p_clip = np.clip(blend, 1e-7, 1.0 - 1e-7)
    ll = -np.mean(np.sum(y_true_oos_oh * np.log(p_clip), axis=1))
    ez = np.dot(p_clip, CLASSES_VEC)
    ic = np.corrcoef(ez, dt_true_oos)[0, 1] if np.std(ez) > 1e-6 else 0.0
    return ll - 0.5 * ic

init_w = np.ones(M) / M
bounds = [(0.0, 1.0) for _ in range(M)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

opt_res = minimize(ensemble_objective, init_w, method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = opt_res.x / np.sum(opt_res.x)

opt_blend = sum(opt_weights[i] * prob_list[i] for i in range(M))
opt_metrics = evaluate_oos_probs(opt_blend)

print("\n" + "=" * 70)
print("🏆 ОПТИМАЛЬНЫЕ ВЕСА МУЛЬТИМОДЕЛЬНОГО АНСАМБЛЯ (W*):")
for name, w in zip(model_names, opt_weights):
    print(f"  • {name:<18}: {w*100:6.2f}%")
print("-" * 70)
print(f"📈 Метрики ансамбля W* vs Одиночный LightGBM:")
print(f"  • OOS IC:          {opt_metrics['ic']:+.4f}  (LightGBM: {m_lgb['ic']:+.4f})")
print(f"  • OOS ROC-AUC:     {opt_metrics['auc']:.4f}  (LightGBM: {m_lgb['auc']:.4f})")
print(f"  • OOS Log-Loss:    {opt_metrics['log_loss']:.4f}  (LightGBM: {m_lgb['log_loss']:.4f})")
print(f"  • Top-2% PnL:      {opt_metrics['top2_pnl']:+.2f} тиков")
print("=" * 70)

🤝 1. Комбинаторный поиск лучших связок из 9 моделей (по методу ensemble_predictor.py)...

ТОП-10 РАВНОВЕСНЫХ СВЯЗОК (по OOS IC):
                                                                  alliance  k       ic      auc  log_loss  top2_pnl
             lightgbm + conv1d+gru_top3 + causal_cnn + mlp_bottleneck_top3  4 0.168623 0.812468  1.361337  0.033654
                              lightgbm + conv1d+gru_top3 + causal_cnn_top3  3 0.167911 0.814084  1.312989  0.043269
                          lightgbm + conv1d+gru_top3 + mlp_bottleneck_top3  3 0.167783 0.814129  1.416626  0.004808
                                                lightgbm + causal_cnn_top3  2 0.167742 0.815732  1.212199  0.057692
        lightgbm + conv1d+gru_top3 + causal_cnn_top3 + mlp_bottleneck_top3  4 0.167733 0.816393  1.388425  0.033654
                          lightgbm + causal_cnn_top3 + mlp_bottleneck_top3  3 0.167669 0.819106  1.334948  0.052885
lightgbm + conv1d+gru + conv1d+gru_top3 + causal_cnn + mlp_

In [8]:
# 8. Упаковка всех обученных моделей, паспортов и конфигурации ансамбля для торгового бота
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
os.makedirs(PACK_DIR, exist_ok=True)

for arch, k_path in trained_keras_models.items():
    arch_dir = os.path.join(PACK_DIR, f"{arch.replace('+', '_')}_h{HORIZON}s")
    os.makedirs(arch_dir, exist_ok=True)
    target_keras = os.path.join(arch_dir, "model_win_00.keras")
    import shutil
    shutil.copy2(k_path, target_keras)

    arch_metrics = evaluate_oos_probs(oos_predictions[arch])
    meta = {
        "model_type": arch,
        "coin": "PRL",
        "H": int(HORIZON),
        "seq_len": SEQ_LEN,
        "train_ts_range": [str(ts_tr[0])[:19], str(ts_tr[-1])[:19]],
        "test_ts_range": [str(ts_te[0])[:19], str(ts_te[-1])[:19]],
        "metrics_oos": {
            "ic_dt_ticks": round(arch_metrics["ic"], 4),
            "roc_auc_ovr": round(arch_metrics["auc"], 4),
            "log_loss": round(arch_metrics["log_loss"], 4),
            "top2pct_mean_ticks": round(arch_metrics["top2_pnl"], 2)
        },
        "feature_norm": {"mean": mean.tolist(), "std": std.tolist()},
        "weights_file": "model_win_00.keras"
    }
    with open(os.path.join(arch_dir, "meta_win_00.json"), "w") as f:
        json.dump(meta, f, indent=2)

alliance_cfg = {
    "coin": "PRL",
    "H": int(HORIZON),
    "seq_len": SEQ_LEN,
    "optimal_weights": {name: round(float(w), 4) for name, w in zip(model_names, opt_weights)},
    "ensemble_metrics_oos": {k: round(float(v), 4) for k, v in opt_metrics.items()},
    "leaderboard": leaderboard
}
with open(os.path.join(PACK_DIR, "ensemble_alliance.json"), "w") as f:
    json.dump(alliance_cfg, f, indent=2)

ZIP_NAME = f"{PACK_DIR}.zip"
!zip -r -q {ZIP_NAME} {PACK_DIR}/

print(f"📦 Архив {ZIP_NAME} готов! Ссылки для скачивания в локальный проект:")
print("🚀 Вариант 1 (Catbox Litterbox):")
!curl -s -F "reqtype=fileupload" -F "time=24h" -F "fileToUpload=@{ZIP_NAME}" https://litterbox.catbox.moe/resources/internals/api.php
print("\n🚀 Вариант 2 (tmpfiles.org):")
!curl -s -F "file=@{ZIP_NAME}" https://tmpfiles.org/api/v1/upload

try:
    from google.colab import files
    files.download(ZIP_NAME)
except Exception:
    pass


📦 Архив prl_ensemble_pack_h5s.zip готов! Ссылки для скачивания в локальный проект:
🚀 Вариант 1 (Catbox Litterbox):
<!doctype html><html lang="en"><meta charset="UTF-8"><meta content="nofollow,noarchive,noindex" name="robots"><title>500 | Internal Server Error</title><meta content="width=device-width,initial-scale=1" name="viewport"><meta content="IE=edge" http-equiv="X-UA-Compatible"><meta content="500 | Internal Server Error" name="title"><meta content="The request was not completed. The server met an unexpected condition." name="description"><meta content="500 | Internal Server Error" property="og:title"><meta content="The request was not completed. The server met an unexpected condition." property="og:description"><meta content="500 | Internal Server Error" property="twitter:title"><meta content="The request was not completed. The server met an unexpected condition." property="twitter:description"><link href="data:image/svg+xml, %3Csvg version='1.0' xmlns='http://www.w3.org/2000/svg

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>